In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Chatbot: Hello! I am your AI assistant.")
print("Type 'exit' or 'quit' to stop.")

chat_history_ids = None

# predefined correct answers
knowledge_base = {
    "who created python": "Python was created by Guido van Rossum.",
    "what is artificial intelligence": "Artificial Intelligence is the simulation of human intelligence by machines.",
    "who are you": "I am an AI chatbot created using Hugging Face Transformers.",
    "what is python": "Python is a high-level programming language used for AI, web, and software development."
}

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    # first check knowledge base
    question = user_input.lower().strip()

    if question in knowledge_base:
        print("Chatbot:", knowledge_base[question])
        continue

    # otherwise use DialoGPT
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors="pt")

    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant.
Type 'exit' or 'quit' to stop.


You:  Hello


Chatbot: Hello! :D


You:  What is Artificial Intelligence


Chatbot: Artificial Intelligence is the simulation of human intelligence by machines.


...

Chatbot: Python was created by Guido van Rossum.


You:  Thankyou


Chatbot: You're welcome! :D


You:  exit


Chatbot: Goodbye!
